<a href="https://colab.research.google.com/github/NeuroNex-ModelX/drainage-failure-sri-lanka/blob/main/04_human_factors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 · Human factors per DS division (census, tree-cover loss, drains)
**Owner:** Prameshka · **Needs:** notebook 00 · **Makes:** `data/processed/human_factors_ds.csv`

**Before running, put these files in place:**
| File | Where it comes from |
|---|---|
| `data/manual/census_ds.csv` | Census 2024 tables (statistics.gov.lk), built by `tools/build_census_ds.py` from the GN population PDF + tables A13, A14, A16 |
| `data/manual/census_gn.csv` *(optional)* | GN-level toilet / water / cooking-fuel counts; summed to DS here |
| `data/raw/gfw/*.csv` | Global Forest Watch ▸ Sri Lanka dashboard ▸ Download (tree cover loss, subnational) |
| `data/raw/osm/` | Geofabrik `sri-lanka-latest-free.shp.zip` (downloaded automatically in Colab) |

Each part is optional: if a file is missing that factor is skipped and Model B uses the rest.

In [1]:
# --- SETUP ---
import os, sys, zipfile
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MODELX_BASE'] = '/content/drive/MyDrive/ModelX'
sys.path.append(os.path.join(os.environ.get('MODELX_BASE', '..'), 'src'))
from common import *

ds_list = load_ds_list()
hf = ds_list[['pcode', 'ds_name', 'district_name', 'area_km2']].copy()

Mounted at /content/drive


## A. Census (population, households, urban share, services)
The first run creates `data/manual/census_ds.csv` with one row per division and empty columns.
Fill what you have (Google Sheets → download as CSV works well), then re-run.

* `population_2024`, `households_2024` – Census 2024, by DS division
* `population_2012` – Census 2012 (for growth), optional
* `urban_share` – share of population in urban sector (0–1)
* `hh_piped_water_share`, `hh_toilet_in_unit_share`, `hh_clean_cooking_share` – service coverage (0–1)

In [2]:
census_path = os.path.join(MANUAL, 'census_ds.csv')
CENSUS_COLS = ['population_2024', 'households_2024', 'population_2012', 'urban_share',
               'hh_piped_water_share', 'hh_toilet_in_unit_share', 'hh_flush_toilet_share', 'hh_clean_cooking_share']
if not os.path.exists(census_path):
    tmpl = ds_list[['pcode', 'district_name', 'ds_name']].copy()
    for c in CENSUS_COLS:
        tmpl[c] = ''
    tmpl.to_csv(census_path, index=False)
    print('Template created ->', census_path, '\nFill it in and re-run this notebook.')

census = pd.read_csv(census_path)
census.columns = [c.strip() for c in census.columns]
if 'pcode' not in census or census['pcode'].isna().any():   # rows typed by name only -> match them
    census = match_ds(census, 'district_name', 'ds_name', 'ds_lookup_census.csv')
    census = census[census['pcode'].notna()]

# Optional GN-level counts -> sum to DS and turn into shares
gn_path = os.path.join(MANUAL, 'census_gn.csv')
if os.path.exists(gn_path):
    gn = pd.read_csv(gn_path)
    gn = match_ds(gn, 'district_name', 'ds_name', 'ds_lookup_census.csv')
    count_cols = [c for c in gn.columns if c.startswith('hh_') and not c.endswith('_share')]
    s = gn.groupby('pcode')[count_cols + ['households']].sum()
    for c in count_cols:
        s[c + '_share'] = s[c] / s['households']
    census = census.drop(columns=[c + '_share' for c in count_cols if c + '_share' in census], errors='ignore')
    census = census.merge(s[[c + '_share' for c in count_cols]].reset_index(), on='pcode', how='left')

census = census.drop(columns=['district_name', 'ds_name'], errors='ignore')
hf = hf.merge(census, on='pcode', how='left')
if 'population_2024' in hf:
    cov = hf['population_2024'].notna()
    print(f'Census coverage: {cov.sum()} of {len(hf)} DS divisions have census data')
    if (~cov).any():
        print('   no census row for:', hf.loc[~cov, 'ds_name'].tolist(),
              '\n   (divisions renamed/split since 2022 - add their pcode to census_ds.csv by hand if you can)')

for c in CENSUS_COLS:
    if c in hf:
        hf[c] = pd.to_numeric(hf[c], errors='coerce')
if 'population_2024' in hf:
    hf['pop_density'] = hf['population_2024'] / hf['area_km2']
if {'population_2024', 'population_2012'} <= set(hf.columns):
    hf['pop_growth_2012_2024'] = hf['population_2024'] / hf['population_2012'] - 1
if 'urban_share' in hf:
    # Census 2024 Table 11.17: 82% of urban vs 12% of rural households get local-authority
    # waste collection -> estimated waste-collection coverage for each division
    hf['est_waste_collection'] = 0.82 * hf['urban_share'] + 0.12 * (1 - hf['urban_share'])
print(hf[[c for c in hf.columns if c not in ('ds_name', 'district_name')]].notna().mean().round(2))

Census coverage: 329 of 339 DS divisions have census data
   no census row for: ['Mathurata', 'Kothmale West', 'Walapane', 'Thalawakele', 'Norwood', 'Wanduramba', 'Madampagama', 'Rathgama', 'Eravur Town', 'Kalthota'] 
   (divisions renamed/split since 2022 - add their pcode to census_ds.csv by hand if you can)
pcode                      1.00
area_km2                   1.00
population_2024            0.97
households_2024            0.97
hh_piped_water_share       0.97
hh_toilet_in_unit_share    0.97
hh_clean_cooking_share     0.97
pop_density                0.97
dtype: float64


## B. Tree-cover loss (Global Forest Watch)
Uses the 30% canopy threshold (GFW default). Newer GFW downloads give loss per **DS division**
(subnational1 = district, subnational2 = DS division) and are matched by name; older ones give it per
district, in which case every DS division in a district gets the district's value.

In [3]:
gfw_tables = []
for f in find_files(os.path.join(RAW, 'gfw'), [r'\.csv$', r'\.xlsx?$']):
    if f.lower().endswith('.csv'):
        gfw_tables.append((os.path.basename(f), pd.read_csv(f)))
    else:   # GFW Excel downloads have one sheet per level ('Subnational 2 tree cover loss', ...)
        for sheet, t in pd.read_excel(f, sheet_name=None).items():
            gfw_tables.append((f'{os.path.basename(f)} [{sheet}]', t))
tree = None
for name, t in gfw_tables:
    t = t.copy()
    t.columns = [str(c).strip().lower() for c in t.columns]
    loss_cols = [c for c in t.columns if re.fullmatch(r'tc_loss_ha_(\d{4})', c)]
    if loss_cols and 'subnational2' in t.columns and 'extent_2000_ha' in t.columns:
        if 'threshold' in t.columns:
            t = t[t['threshold'] == 30]
        t['tree_loss_pct_2001_2024'] = 100 * t[loss_cols].sum(axis=1) / t['extent_2000_ha']
        recent = [c for c in loss_cols if int(c[-4:]) >= 2015]
        t['tree_loss_pct_2015_2024'] = 100 * t[recent].sum(axis=1) / t['extent_2000_ha']
        t['tree_cover_2000_pct'] = 100 * t['extent_2000_ha'] / t['area_ha']
        keep = ['subnational1', 'subnational2', 'tree_loss_pct_2001_2024', 'tree_loss_pct_2015_2024', 'tree_cover_2000_pct']
        tree = t[[c for c in keep if c in t.columns]].copy()
        GFW_YEARS = f'{loss_cols[0][-4:]}-{loss_cols[-1][-4:]}'
        print('Used', name, '| years', GFW_YEARS)
        break
if tree is None and gfw_tables:
    print('GFW files found but none has subnational2 + tc_loss_ha_YYYY columns:', [n for n, _ in gfw_tables])

if tree is not None:
    TREE_COLS = ['tree_loss_pct_2001_2024', 'tree_loss_pct_2015_2024', 'tree_cover_2000_pct']
    # Newer GFW downloads use GADM 4: subnational1 = district, subnational2 = DS division.
    # Older ones: subnational2 = district. Decide by how many distinct names there are.
    if tree['subnational2'].nunique() > 1.5 * ds_list['district_name'].nunique() and 'subnational1' in tree:
        level = 'DS division'
        tree = match_ds(tree, 'subnational1', 'subnational2', 'ds_lookup_gfw.csv')
        tree = tree[tree['pcode'].notna()].groupby('pcode', as_index=False)[TREE_COLS].mean()
        hf = hf.merge(tree, on='pcode', how='left')
    else:
        level = 'district'
        dmap = {norm_name(d): d for d in ds_list['district_name'].unique()}
        tree['district_name'] = tree['subnational2'].map(lambda x: dmap.get(norm_name(x)))
        if tree['district_name'].isna().any():
            print('Unmatched GFW districts:', tree.loc[tree['district_name'].isna(), 'subnational2'].tolist())
        hf = hf.merge(tree.dropna(subset=['district_name'])[['district_name'] + TREE_COLS],
                      on='district_name', how='left')
    print(f"Tree loss attached at {level} level: {hf['tree_loss_pct_2001_2024'].notna().sum()} of {len(hf)} divisions")
    save_meta('tree_loss', f'Global Forest Watch / Hansen et al., tree cover loss {GFW_YEARS}, by {level}')
else:
    print('No usable GFW file found - tree loss skipped.')

Used LKA.xlsx [Subnational 2 tree cover loss] | years 2001-2025
24 names could not be matched automatically.
   -> open data/manual/ds_lookup_gfw.csv, fill the 'pcode' column (see data/processed/ds_list.csv), re-run.
Matched 92.6% of rows to a DS division.
Tree loss attached at DS division level: 299 of 339 divisions


## C. Drain and canal density (OpenStreetMap waterways)
Kilometres of mapped drains + canals per km², and of streams + rivers per km².
(OSM mapping is better in cities - mention this as a data gap in the report.)

In [4]:
import subprocess
osm_dir = os.path.join(RAW, 'osm')
os.makedirs(osm_dir, exist_ok=True)
zip_path = os.path.join(osm_dir, 'sri-lanka-latest-free.shp.zip')
URL = 'https://download.geofabrik.de/asia/sri-lanka-latest-free.shp.zip'


def zip_ok(p):
    try:
        with zipfile.ZipFile(p) as zf:
            return zf.testzip() is None
    except Exception:
        return False


if not find_files(osm_dir, [r'waterways.*\.shp$']):          # not extracted yet
    if os.path.exists(zip_path) and not zip_ok(zip_path):
        print('Broken zip found - deleting and downloading again')
        os.remove(zip_path)
    if not os.path.exists(zip_path) and IN_COLAB:
        tmp = '/content/osm.zip'                                # download to Colab disk first (faster, safer)
        subprocess.run(['wget', '-q', '--tries=3', URL, '-O', tmp])
        if zip_ok(tmp):
            subprocess.run(['cp', tmp, zip_path])
            print(f'Downloaded OSM extract ({os.path.getsize(zip_path) / 1e6:.0f} MB)')
        else:
            print('Download failed or incomplete - download it in your browser from', URL,
                  'and upload it to data/raw/osm/')
    if os.path.exists(zip_path) and zip_ok(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            for n in zf.namelist():
                if 'waterways' in n:
                    zf.extract(n, osm_dir)
        print('Extracted waterways layer')
ww_files = find_files(osm_dir, [r'waterways.*\.shp$'])
if ww_files:
    import geopandas as gpd
    ww = gpd.read_file(ww_files[0])[['fclass', 'geometry']].to_crs(SL_CRS)
    poly = gpd.read_file(os.path.join(PROC, 'ds_divisions.gpkg'))[['pcode', 'geometry']].to_crs(SL_CRS)
    # Fast: give each waterway to the division containing its midpoint (gpd.overlay on detailed
    # boundaries takes 30+ min; OSM lines are short, so the effect on density is negligible)
    ww = ww[ww['fclass'].isin(['drain', 'canal', 'ditch', 'stream', 'river'])].copy()
    ww['km'] = ww.length / 1000
    mid = ww.copy()
    mid['geometry'] = ww.geometry.interpolate(0.5, normalized=True)
    pieces = gpd.sjoin(mid, poly, predicate='within', how='inner')
    print(f'{len(pieces):,} waterway segments assigned to DS divisions')
    lens = pieces.pivot_table(index='pcode', columns='fclass', values='km', aggfunc='sum', fill_value=0)
    out = pd.DataFrame(index=lens.index)
    out['drain_canal_km'] = lens.reindex(columns=['drain', 'canal', 'ditch'], fill_value=0).sum(axis=1)
    out['stream_river_km'] = lens.reindex(columns=['stream', 'river'], fill_value=0).sum(axis=1)
    hf = hf.merge(out.reset_index(), on='pcode', how='left')
    hf[['drain_canal_km', 'stream_river_km']] = hf[['drain_canal_km', 'stream_river_km']].fillna(0)
    hf['drain_density'] = hf['drain_canal_km'] / hf['area_km2']
    hf['stream_density'] = hf['stream_river_km'] / hf['area_km2']
    save_meta('drains', 'OpenStreetMap waterways (Geofabrik extract), ODbL')
else:
    print('No OSM waterways file found - drain density skipped (first thing to cut, per proposal).')

Broken zip found - deleting and downloading again
Downloaded OSM extract (365 MB)
Extracted waterways layer
16,924 waterway segments assigned to DS divisions


In [5]:
hf.to_csv(os.path.join(PROC, 'human_factors_ds.csv'), index=False)
save_meta('census', 'Census of Population and Housing 2024 (DCS): population (GN table summed to DS), households, '
                    'piped water (A14), in-house toilet (A16), clean cooking fuel (A13)'
                    + ('; waste collection estimated from urban share' if 'urban_share' in hf and hf['urban_share'].notna().any() else ''))
print('Saved human_factors_ds.csv with columns:', [c for c in hf.columns if c not in ('ds_name', 'district_name')])

Saved human_factors_ds.csv with columns: ['pcode', 'area_km2', 'population_2024', 'households_2024', 'hh_piped_water_share', 'hh_toilet_in_unit_share', 'hh_clean_cooking_share', 'pop_density', 'tree_loss_pct_2001_2024', 'tree_loss_pct_2015_2024', 'tree_cover_2000_pct', 'drain_canal_km', 'stream_river_km', 'drain_density', 'stream_density']
